In [13]:
import pandas as pd
import geopandas as gpd


In [14]:
era = pd.read_csv("ERA5_2025.csv")
print(era.head())

print("\nShape:", era.shape)

print("\nColumns:")
print(era.columns)

print("\nMissing values:")
print(era.isnull().sum())

                  time  latitude  longitude  number    step  surface  \
0  2025-01-01 00:00:00      22.0      72.75       0  0 days      0.0   
1  2025-01-01 00:00:00      22.0      73.00       0  0 days      0.0   
2  2025-01-01 00:00:00      22.0      73.25       0  0 days      0.0   
3  2025-01-01 00:00:00      22.0      73.50       0  0 days      0.0   
4  2025-01-01 00:00:00      22.0      73.75       0  0 days      0.0   

            valid_time       u10       v10        t2m         sp       blh  
0  2025-01-01 00:00:00 -0.765518 -1.679825  290.61720  101349.94  80.32663  
1  2025-01-01 00:00:00 -0.942276 -1.389786  290.28320  101205.94  62.32663  
2  2025-01-01 00:00:00 -1.162003 -1.254044  290.96680  101127.94  52.32663  
3  2025-01-01 00:00:00 -1.858292 -0.723770  291.08984  100673.94  37.26413  
4  2025-01-01 00:00:00 -1.507706  0.398300  289.99220   99696.94  42.01413  

Shape: (7516080, 12)

Columns:
Index(['time', 'latitude', 'longitude', 'number', 'step', 'surface',
    

In [15]:
era = era.dropna(how="all")
era = era.dropna(axis=1, how="all")


In [16]:
# ==========================
# 3. DATETIME CLEANING
# ==========================

era["valid_time"] = pd.to_datetime(
    era["valid_time"],
    errors="coerce"
)

era["date"] = era["valid_time"].dt.date
era["time"] = era["valid_time"].dt.time

In [17]:
drop_cols = [
    "number",
    "step",
    "surface",
    "valid_time"
]

drop_cols = [
    c for c in drop_cols
    if c in era.columns
]

era = era.drop(columns=drop_cols)


In [ ]:
# ==========================
# 5. LAT-LON TO DISTRICT
# ==========================

districts = gpd.read_file("gadm41_IND_2.shp")

# Keep only Maharashtra districts
districts = districts[
    districts["NAME_1"] == "Maharashtra"
]

era_gdf = gpd.GeoDataFrame(
    era,
    geometry=gpd.points_from_xy(
        era.longitude,
        era.latitude
    ),
    crs="EPSG:4326"
)

era_district = gpd.sjoin(
    era_gdf,
    districts,
    how="left",
    predicate="within"
)

district_col = "NAME_2"

era_district["district"] = era_district["NAME_2"]

In [ ]:
# ==========================
# 6. DISTRICT AGGREGATION
# ==========================

era_clean = (
    era_district
    .groupby(
        ["district", "date"],
        as_index=False
    )
    .agg({
        "u10": "mean",
        "v10": "mean",
        "t2m": "mean",
        "sp": "mean",
        "blh": "mean"
    })
)

In [ ]:
# ==========================
# 7. FINAL ANALYSIS
# ==========================

print("\n======================")
print("CLEANED DATASET")
print("======================")

print(era_clean.head())

print("\nShape:", era_clean.shape)

print("\nMissing values:")
print(era_clean.isnull().sum())


In [ ]:
era_clean.to_csv(
    "ERA_2025_clean.csv",
    index=False
)

print("\nSaved: ERA_2025_clean.csv")